# Notebook 11 - Processes gentrified industries

In [1]:
import os
import json
import pandas as pd
import os

### Load data

In [2]:
with open("../data/processed/cvr/CVR_geo_temporal.geojson", "r", encoding="utf-8") as f:
    cvr_geo = json.load(f)

print(f"Top-level keys: {cvr_geo.keys()}")
print(f"Number of features: {len(cvr_geo['features'])}")
print(json.dumps(cvr_geo["features"][0], indent=2, ensure_ascii=False))

Top-level keys: dict_keys(['type', 'features'])
Number of features: 332071
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      12.59552089,
      55.69984689
    ]
  },
  "properties": {
    "CVRNummer": 13412731,
    "navn": "CARNEGIE ASSET ADMINISTRATION A/S",
    "virksomhedStartdato": "1989-08-01",
    "virksomhedOphoersdato": "2019-06-11",
    "branche_kode": "661900",
    "branche_navn": "Andre hjælpetjenester i forbindelse med finansiel formidling",
    "CVREnhedsId": "4002799827",
    "Adresse": "d8f92e9d-db74-45b0-89b2-973686e16809",
    "pNummer": 1000558874,
    "tilknyttetVirksomhedsCVRNummer": 13412731,
    "produktionsenhedStartdato": "1989-08-01",
    "produktionsenhedOphoersdato": "2019-06-11"
  }
}


In [3]:
filepath = "../data/raw/cvr_raw/industry_ids/Gentrified_business_total_industry_id.csv"
#filepath = "../data/raw/cvr_raw/industry_ids/Gentrified_business_certain_industry_id.csv"
gentrified_industries_df = pd.read_csv(filepath, encoding="cp1252", sep=";")
gentrified_industries_df.head()

,industry-id,name
0,471110,Detailhandel med kioskvarer
1,471200,Anden ikke-specialiseret detailhandel
2,472100,Detailhandel med frugt og grï¿½ntsager
3,472200,Detailhandel med kï¿½d og kï¿½dprodukter
4,472300,"Detailhandel med fisk, krebsdyr og blï¿½ddyr"


In [4]:
#change industry-id to branche_kode
gentrified_industries_df.rename(columns={"industry-id": "branche_kode"}, inplace=True)
gentrified_industries_df.head() 

,branche_kode,name
0,471110,Detailhandel med kioskvarer
1,471200,Anden ikke-specialiseret detailhandel
2,472100,Detailhandel med frugt og grï¿½ntsager
3,472200,Detailhandel med kï¿½d og kï¿½dprodukter
4,472300,"Detailhandel med fisk, krebsdyr og blï¿½ddyr"


### Filter by branche_kode (industry id)

filter all object to only include objects with branche id listed in the gentrified industry list

In [5]:
# Build a set of allowed branche_kode values (as strings for safe comparison)
valid_codes = set(gentrified_industries_df["branche_kode"].astype(str))

# Filter GeoJSON features to only those with a matching branche_kode
filtered_features = [
    f for f in cvr_geo["features"]
    if str(f["properties"].get("branche_kode", "")) in valid_codes
]

print(f"Before: {len(cvr_geo['features'])} features")
print(f"After:  {len(filtered_features)} features")

# Replace features in the geo dict
cvr_geo["features"] = filtered_features

Before: 332071 features
After:  15403 features


In [6]:
#show one example of the filtered features
if cvr_geo["features"]:
    print(json.dumps(cvr_geo["features"][0], indent=2, ensure_ascii=False))
    

{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      12.49925711,
      55.70431187
    ]
  },
  "properties": {
    "CVRNummer": 62756217,
    "navn": "SYNOPTIK A/S",
    "virksomhedStartdato": "1941-06-28",
    "virksomhedOphoersdato": null,
    "branche_kode": "477410",
    "branche_navn": "Optikeraktiviteter",
    "CVREnhedsId": "4002190707",
    "Adresse": "940308e1-5e4d-49d3-a939-5c05e8efee5f",
    "pNummer": 1003150905,
    "tilknyttetVirksomhedsCVRNummer": 62756217,
    "produktionsenhedStartdato": "1977-07-04",
    "produktionsenhedOphoersdato": null
  }
}


## Mapping gentrified industried

A timeline show when i gentried buiness open and when it closed

In [7]:
import folium
from folium.plugins import TimestampedGeoJson, MarkerCluster

# Build a DataFrame from filtered features
rows = []
for feat in cvr_geo["features"]:
    props = feat["properties"]
    coords = feat["geometry"]["coordinates"]  # [lon, lat]
    rows.append({
        "lon": coords[0],
        "lat": coords[1],
        "navn": props.get("navn", ""),
        "branche_navn": props.get("branche_navn", ""),
        "start": props.get("produktionsenhedStartdato"),
        "end": props.get("produktionsenhedOphoersdato"),
    })

df = pd.DataFrame(rows)
df["start"] = pd.to_datetime(df["start"])
df["end"] = pd.to_datetime(df["end"])  # NaT means still open

# Clamp timeline to 2000-2026 to keep the slider manageable
# Businesses that started before 2000 and are still active will appear from 2000 onward
YEAR_START = 2000
YEAR_END = 2026

# --- Timeline layer ---
ts_features = []
for _, row in df.iterrows():
    sy = max(row["start"].year, YEAR_START)
    ey = row["end"].year if pd.notna(row["end"]) else YEAR_END
    ey = min(ey, YEAR_END)
    if sy > ey:
        continue  # business closed before YEAR_START
    still_open = pd.isna(row["end"])
    color = "#2ca02c" if still_open else "#d62728"

    for y in range(sy, ey + 1):
        ts_features.append({
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [row["lon"], row["lat"]],
            },
            "properties": {
                "times": [f"{y}-01-01T00:00:00"],
                "popup": (
                    f"<b>{row['navn']}</b><br>"
                    f"{row['branche_navn']}<br>"
                    f"Opened: {row['start'].date()}<br>"
                    f"Closed: {row['end'].date() if not still_open else 'Still open'}"
                ),
                "icon": "circle",
                "iconstyle": {
                    "fillColor": color,
                    "fillOpacity": 0.8,
                    "stroke": "true",
                    "color": color,
                    "radius": 7,
                },
            },
        })

print(f"Timeline features: {len(ts_features)} (from {YEAR_START} to {YEAR_END})")
print(f"Businesses: {len(df)}")

# Build map
center = [df["lat"].mean(), df["lon"].mean()]
m = folium.Map(location=center, zoom_start=12, tiles="cartodbpositron")

TimestampedGeoJson(
    {"type": "FeatureCollection", "features": ts_features},
    period="P1Y",
    duration="P1Y",
    auto_play=False,
    loop=False,
    max_speed=5,
    loop_button=True,
    date_options="YYYY",
    time_slider_drag_update=True,
    transition_time=300,
).add_to(m)

# --- Cluster layer (toggle via layer control) ---
cluster = MarkerCluster(name="All businesses (clustered)", show=False)
for _, row in df.iterrows():
    still_open = pd.isna(row["end"])
    color = "green" if still_open else "red"
    popup_text = (
        f"<b>{row['navn']}</b><br>"
        f"{row['branche_navn']}<br>"
        f"Opened: {row['start'].date()}<br>"
        f"Closed: {row['end'].date() if not still_open else 'Still open'}"
    )
    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=folium.Popup(popup_text, max_width=300),
        icon=folium.Icon(color=color, icon="info-sign"),
    ).add_to(cluster)
cluster.add_to(m)

folium.LayerControl().add_to(m)

# Legend
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
    background:white; padding:10px; border:2px solid grey; border-radius:5px;">
<b>Legend</b><br>
<i style="background:#2ca02c;width:12px;height:12px;display:inline-block;border-radius:50%;"></i> Still open<br>
<i style="background:#d62728;width:12px;height:12px;display:inline-block;border-radius:50%;"></i> Closed
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

out_path = os.path.join("..", "results", "figures", "gentrified_industries_timeline.html")
m.save(out_path)
print(f"Saved to {out_path}")
#m

Timeline features: 141061 (from 2000 to 2026)
Businesses: 15403
Saved to ..\results\figures\gentrified_industries_timeline.html


## Moving hotspot map: aggregate gentrified businesses into cluster polygons over time

Use `clusters_hovedstad_clean.shp` to spatially aggregate gentrified businesses per neighborhood cluster per year, and visualise as an animated choropleth.

In [8]:
import geopandas as gpd
from shapely.geometry import Point
import numpy as np

# Load cluster polygons
clusters = gpd.read_file("../data/raw/GIS_lag/clusters_hovedstad_clean.shp")  # EPSG:25832

# Merge all Frederiksberg clusters into one polygon
# Catches munic_code == 147 (numeric) and munic_code starting with "147_" (string)
fre_mask = clusters["munic_code"].astype(str).str.match(r"^147(\.0)?$|^147_")
fre_rows = clusters[fre_mask]
non_fre = clusters[~fre_mask].copy()

if len(fre_rows) > 0:
    fre_dissolved = gpd.GeoDataFrame(
        {"cluster_id": ["147_all"], "munic_code": [147.0],
         "geometry": [fre_rows.union_all()]},
        crs=clusters.crs,
    )
    clusters = pd.concat([non_fre, fre_dissolved], ignore_index=True)
    print(f"Dissolved {len(fre_rows)} Frederiksberg clusters into 1")

print(f"Total clusters after dissolve: {len(clusters)}")

# Build GeoDataFrame from filtered CVR features (WGS84 points)
biz_rows = []
for feat in cvr_geo["features"]:
    props = feat["properties"]
    lon, lat = feat["geometry"]["coordinates"]
    biz_rows.append({
        "lon": lon,
        "lat": lat,
        "start": props.get("produktionsenhedStartdato"),
        "end": props.get("produktionsenhedOphoersdato"),
        "navn": props.get("navn", ""),
        "branche_navn": props.get("branche_navn", ""),
    })

biz_gdf = gpd.GeoDataFrame(
    biz_rows,
    geometry=[Point(r["lon"], r["lat"]) for r in biz_rows],
    crs="EPSG:4326",
)
biz_gdf["start"] = pd.to_datetime(biz_gdf["start"])
biz_gdf["end"] = pd.to_datetime(biz_gdf["end"])  # NaT = still open

# Reproject business points to match cluster CRS (EPSG:25832)
biz_gdf = biz_gdf.to_crs(clusters.crs)

# Spatial join: assign each business to a cluster
joined = gpd.sjoin(biz_gdf, clusters[["cluster_id", "geometry"]], how="inner", predicate="within")
print(f"Businesses matched to clusters: {len(joined)} / {len(biz_gdf)}")
joined.head(3)

Dissolved 236 Frederiksberg clusters into 1
Total clusters after dissolve: 1186
Businesses matched to clusters: 10569 / 15403


,lon,lat,start,end,navn,branche_navn,geometry,index_right,cluster_id
0,12.499257,55.704312,1977-07-04,NaT,SYNOPTIK A/S,Optikeraktiviteter,POINT (719854.46 6178720.34),886,101_892
1,12.609342,55.655718,1977-07-04,NaT,SYNOPTIK A/S,Optikeraktiviteter,POINT (727049.23 6173670.901),1003,101_1013
3,12.579280,55.679885,1977-07-04,NaT,SYNOPTIK A/S,Optikeraktiviteter,POINT (725020.22 6176260.601),558,101_564


In [9]:
# Count active businesses per cluster per year (3-year steps from 1987, ending at 2026)
YEAR_START, YEAR_END = 1987, 2026
years = list(range(YEAR_START, YEAR_END, 3))
if years[-1] != YEAR_END:
    years.append(YEAR_END)

records = []
for year in years:
    ts = pd.Timestamp(f"{year}-01-01")
    # A business is "active" if it started on or before Jan 1 of that year
    # and has not yet closed (end is NaT or end >= Jan 1 of that year)
    active = joined[
        (joined["start"] <= ts) &
        (joined["end"].isna() | (joined["end"] >= ts))
    ]
    counts = active.groupby("cluster_id").size().reset_index(name="count")
    counts["year"] = year
    records.append(counts)

yearly_counts = pd.concat(records, ignore_index=True)
print(f"Years: {years}")
print(f"Yearly aggregation: {len(yearly_counts)} rows over {len(years)} time steps")
print(f"Clusters with at least 1 business in any year: {yearly_counts['cluster_id'].nunique()}")
yearly_counts.head()

Years: [1987, 1990, 1993, 1996, 1999, 2002, 2005, 2008, 2011, 2014, 2017, 2020, 2023, 2026]
Yearly aggregation: 7568 rows over 14 time steps
Clusters with at least 1 business in any year: 1027


,cluster_id,count,year
0,101_1002,1,1987
1,101_1009,1,1987
2,101_1013,1,1987
3,101_102,1,1987
4,101_1020,1,1987


In [10]:
import folium
from folium.plugins import TimeSliderChoropleth
import branca.colormap as cm
import datetime

# Prepare the animated choropleth data

# Compute center in projected CRS (EPSG:25832) then convert to lat/lon
centroid_proj = clusters.geometry.centroid
center_proj = gpd.GeoSeries([centroid_proj.union_all().centroid], crs=clusters.crs)
center_4326 = center_proj.to_crs("EPSG:4326")
center = [center_4326.y.iloc[0], center_4326.x.iloc[0]]

# Reproject ALL clusters to WGS84 for Folium
clusters_4326 = clusters.to_crs("EPSG:4326").copy()
clusters_4326["feature_id"] = clusters_4326["cluster_id"].astype(str)

# Global max count for color scaling
active_ids = set(yearly_counts["cluster_id"])
max_count = yearly_counts["count"].max()

# Build a colormap (light yellow → orange → dark red)
colormap = cm.LinearColormap(
    colors=["#ffffcc", "#fd8d3c", "#bd0026"],
    vmin=0,
    vmax=max_count,
    caption="Active gentrified businesses",
)

# Pivot yearly_counts to a dict keyed by cluster_id then year
count_lookup = {}
for _, row in yearly_counts.iterrows():
    count_lookup.setdefault(row["cluster_id"], {})[row["year"]] = row["count"]

# Build styledict for ALL clusters
# Clusters without businesses → light gray; Frederiksberg → fully transparent
GRAY = "#d3d3d3"
FRE_ID = "147_all"
styledict = {}
for cid in clusters_4326["feature_id"]:
    year_styles = {}
    is_fre = (cid == FRE_ID)
    for year in years:
        epoch = str(int(datetime.datetime(year, 1, 1).timestamp()))
        if is_fre:
            year_styles[epoch] = {"color": GRAY, "opacity": 0.0}
            continue
        cnt = count_lookup.get(cid, {}).get(year, 0)
        if cnt == 0:
            hex_color = GRAY
        else:
            rgba = colormap.rgba_floats_tuple(cnt)
            hex_color = "#{:02x}{:02x}{:02x}".format(
                int(rgba[0] * 255), int(rgba[1] * 255), int(rgba[2] * 255)
            )
        year_styles[epoch] = {"color": hex_color, "opacity": 0.8}
    styledict[cid] = year_styles

# Build GeoJSON — set feature_id as index for correct feature-level 'id'
geo_data = clusters_4326[["feature_id", "geometry"]].copy()
geo_data = geo_data.set_index("feature_id")
geo_json = geo_data.to_json()

# Create the map
m2 = folium.Map(location=center, zoom_start=12, tiles="cartodbpositron")

# Animated choropleth layer
TimeSliderChoropleth(
    data=geo_json,
    styledict=styledict,
).add_to(m2)

# Static black-border overlay on ALL clusters (no fill, just stroke)
folium.GeoJson(
    clusters_4326[["geometry"]].to_json(),
    style_function=lambda feat: {
        "fillOpacity": 0,
        "color": "black",
        "weight": 0.6,
    },
    name="Cluster borders",
).add_to(m2)

colormap.add_to(m2)

# --- Play / Pause button (custom JS) ---
# The slider is created by D3 *after* the page loads, so we look it up
# lazily when the user clicks Play, not on script parse time.
play_js = """
<style>
#hotspot-play-btn {
    position: fixed; bottom: 24px; left: 50%; transform: translateX(-50%);
    z-index: 10000; padding: 8px 22px; font-size: 15px; cursor: pointer;
    background: white; border: 2px solid #666; border-radius: 5px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.3);
}
</style>
<button id="hotspot-play-btn">&#9654; Play</button>
<script>
(function(){
    var playing = false, timer = null;

    function getSlider() {
        // TimeSliderChoropleth builds the range input inside a div whose id
        // starts with 'slider_macro_element_'
        var container = document.querySelector('[id^="slider_macro_element_"]');
        if (!container) return null;
        return container.querySelector('input[type="range"]');
    }

    document.getElementById('hotspot-play-btn').addEventListener('click', function() {
        var btn = this;
        var slider = getSlider();
        if (!slider) {
            btn.textContent = 'Slider not found';
            return;
        }
        if (playing) {
            clearInterval(timer);
            playing = false;
            btn.innerHTML = '&#9654; Play';
        } else {
            playing = true;
            btn.innerHTML = '&#9724; Pause';
            timer = setInterval(function(){
                var val = parseInt(slider.value);
                var mx  = parseInt(slider.max);
                if (val >= mx) {
                    slider.value = slider.min;
                } else {
                    slider.value = val + 1;
                }
                slider.dispatchEvent(new Event('input', {bubbles: true}));
            }, 800);
        }
    });
})();
</script>
"""
m2.get_root().html.add_child(folium.Element(play_js))

# Save
out_path2 = os.path.join("..", "results", "figures", "gentrified_industries_hotspot.html")
m2.save(out_path2)
print(f"Saved moving hotspot map to {out_path2}")
#m2

Saved moving hotspot map to ..\results\figures\gentrified_industries_hotspot.html


## Map showing gentrified buinesses hotspot: Median and mean values local (neighborhood level)

In [11]:
import folium
from folium.plugins import TimeSliderChoropleth
import datetime
import numpy as np

# --- Compute deviations from MEDIAN and MEAN per cluster ---
all_cluster_ids = clusters_4326["feature_id"].unique()
full_index = pd.MultiIndex.from_product([all_cluster_ids, years], names=["cluster_id", "year"])
full_df = (
    yearly_counts.set_index(["cluster_id", "year"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

medians = full_df.groupby("cluster_id")["count"].median().rename("median")
means = full_df.groupby("cluster_id")["count"].mean().rename("mean")
full_df = full_df.merge(medians, on="cluster_id").merge(means, on="cluster_id")
full_df["dev_median"] = full_df["count"] - full_df["median"]
full_df["dev_mean"] = full_df["count"] - full_df["mean"]

# Build lookups: cluster_id → year → deviation
dev_median_lookup = {}
dev_mean_lookup = {}
for _, row in full_df.iterrows():
    dev_median_lookup.setdefault(row["cluster_id"], {})[row["year"]] = row["dev_median"]
    dev_mean_lookup.setdefault(row["cluster_id"], {})[row["year"]] = row["dev_mean"]

# --- Categorical color scheme ---
BLUE   = "#2166ac"
WHITE  = "#f7f7f7"
YELLOW = "#fee08b"
ORANGE = "#f46d43"
RED    = "#a50026"
GRAY   = "#d3d3d3"

t1 = 1
t2 = 3

def classify_color(dev, has_data):
    if not has_data:
        return GRAY
    if dev < 0:
        return BLUE
    if dev == 0:
        return WHITE
    if dev <= t1:
        return YELLOW
    if dev <= t2:
        return ORANGE
    return RED

def classify_label(dev):
    if dev < 0:
        return "Below"
    if dev == 0:
        return "At"
    if dev <= t1:
        return "Little above"
    if dev <= t2:
        return "Medium above"
    return "High above"

print(f"Thresholds: little ≤ {t1}, medium ≤ {t2}, high > {t2}")

# --- Build styledicts for both metrics, and combined popups ---
FRE_ID = "147_all"
styledict_median = {}
styledict_mean = {}
popup_dict = {}

for cid in clusters_4326["feature_id"]:
    med_styles = {}
    mn_styles = {}
    is_fre = (cid == FRE_ID)
    has_med = cid in dev_median_lookup
    has_mn = cid in dev_mean_lookup
    popup_rows = []

    for year in years:
        epoch = str(int(datetime.datetime(year, 1, 1).timestamp()))
        if is_fre:
            med_styles[epoch] = {"color": GRAY, "opacity": 0.0}
            mn_styles[epoch] = {"color": GRAY, "opacity": 0.0}
            continue

        d_med = dev_median_lookup.get(cid, {}).get(year, 0.0)
        d_mn = dev_mean_lookup.get(cid, {}).get(year, 0.0)

        med_styles[epoch] = {"color": classify_color(d_med, has_med), "opacity": 0.8}
        mn_styles[epoch] = {"color": classify_color(d_mn, has_mn), "opacity": 0.8}

        cnt = full_df[(full_df["cluster_id"] == cid) & (full_df["year"] == year)]["count"].values[0]
        cat_med = classify_label(d_med) if has_med else "N/A"
        cat_mn = classify_label(d_mn) if has_mn else "N/A"
        popup_rows.append(
            f"<tr><td>{year}</td><td>{cnt}</td>"
            f"<td>{d_med:.1f}</td><td>{cat_med}</td>"
            f"<td>{d_mn:.1f}</td><td>{cat_mn}</td></tr>"
        )

    styledict_median[cid] = med_styles
    styledict_mean[cid] = mn_styles

    med_val = full_df[full_df["cluster_id"] == cid]["median"].iloc[0]
    mn_val = full_df[full_df["cluster_id"] == cid]["mean"].iloc[0]
    popup_html = (
        f"<b>Neighborhood: {cid}</b><br>"
        f"<b>Median: {med_val:.1f} &nbsp;|&nbsp; Mean: {mn_val:.1f}</b>"
        f"<table border='1' style='border-collapse:collapse;font-size:11px;'>"
        f"<tr><th>Year</th><th>Count</th>"
        f"<th>Dev(Med)</th><th>Cat(Med)</th>"
        f"<th>Dev(Mean)</th><th>Cat(Mean)</th></tr>"
        f"{''.join(popup_rows)}</table>"
    )
    popup_dict[cid] = popup_html

# --- Build GeoJSON ---
geo_data_dev = clusters_4326[["feature_id", "geometry"]].copy()
geo_data_dev = geo_data_dev.set_index("feature_id")
geo_json_dev = geo_data_dev.to_json()

# --- Create map ---
m3 = folium.Map(location=center, zoom_start=12, tiles="cartodbpositron")

# Two TimeSliderChoropleth layers: median (visible), mean (hidden)
tsc_median = TimeSliderChoropleth(
    data=geo_json_dev, styledict=styledict_median,
    name="Median deviation", show=True,
)
tsc_median.add_to(m3)

tsc_mean = TimeSliderChoropleth(
    data=geo_json_dev, styledict=styledict_mean,
    name="Mean deviation", show=False,
)
tsc_mean.add_to(m3)

# Get JS variable names for toggle logic
median_var = tsc_median.get_name()
mean_var = tsc_mean.get_name()
map_var = m3.get_name()

# Popup layer (always visible, transparent fill, black borders, click for info)
popup_gdf = clusters_4326[["feature_id", "geometry"]].copy()
popup_gdf["popup_html"] = popup_gdf["feature_id"].map(popup_dict)

folium.GeoJson(
    json.loads(popup_gdf.to_json()),
    style_function=lambda feat: {
        "fillOpacity": 0,
        "color": "black",
        "weight": 0.6,
    },
    popup=folium.GeoJsonPopup(
        fields=["popup_html"],
        aliases=[""],
        labels=False,
        parse_html=True,
        max_width=500,
    ),
    name="Cluster borders",
).add_to(m3)

# --- Categorical legend ---
legend_html = f"""
<div id="dev-legend" style="position:fixed; bottom:30px; left:30px; z-index:1000;
    background:white; padding:10px 14px; border:2px solid grey; border-radius:5px;
    font-size:13px; line-height:1.6;">
<b id="legend-title">Deviation from median</b><br>
<i style="background:{BLUE};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> Below<br>
<i style="background:{WHITE};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> At<br>
<i style="background:{YELLOW};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> Little above (≤ 1)<br>
<i style="background:{ORANGE};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> Medium above (1–3)<br>
<i style="background:{RED};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> High above (> 3)<br>
<i style="background:{GRAY};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> No data
</div>
"""
m3.get_root().html.add_child(folium.Element(legend_html))

# --- Metric toggle (radio buttons) ---
toggle_html = f"""
<div id="metric-toggle" style="position:fixed; top:10px; right:10px; z-index:10000;
    background:white; padding:10px 14px; border:2px solid #666; border-radius:5px;
    box-shadow:0 2px 6px rgba(0,0,0,0.3); font-size:14px;">
<b>Metric:</b><br>
<label style="cursor:pointer;"><input type="radio" name="metric" value="median" checked> Median</label><br>
<label style="cursor:pointer;"><input type="radio" name="metric" value="mean"> Mean</label>
</div>
<script>
document.querySelectorAll('input[name="metric"]').forEach(function(radio) {{
    radio.addEventListener('change', function() {{
        var legendTitle = document.getElementById('legend-title');
        if (this.value === 'median') {{
            {map_var}.removeLayer({mean_var});
            {median_var}.addTo({map_var});
            if (legendTitle) legendTitle.textContent = 'Deviation from median';
        }} else {{
            {map_var}.removeLayer({median_var});
            {mean_var}.addTo({map_var});
            if (legendTitle) legendTitle.textContent = 'Deviation from mean';
        }}
    }});
}});
</script>
"""
m3.get_root().html.add_child(folium.Element(toggle_html))

# --- Play / Pause button ---
play_js_dev = f"""
<style>
#median-play-btn {{
    position: fixed; bottom: 24px; left: 50%; transform: translateX(-50%);
    z-index: 10000; padding: 8px 22px; font-size: 15px; cursor: pointer;
    background: white; border: 2px solid #666; border-radius: 5px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.3);
}}
</style>
<button id="median-play-btn">&#9654; Play</button>
<script>
(function(){{
    var playing = false, timer = null;

    function getActiveSlider() {{
        // Find the visible slider (the active layer's slider)
        var sliders = document.querySelectorAll('[id^="slider_"]');
        for (var i = 0; i < sliders.length; i++) {{
            if (sliders[i].offsetParent !== null) {{
                return sliders[i].querySelector('input[type="range"]');
            }}
        }}
        return null;
    }}

    document.getElementById('median-play-btn').addEventListener('click', function() {{
        var btn = this;
        var slider = getActiveSlider();
        if (!slider) {{
            btn.textContent = 'Slider not found';
            return;
        }}
        if (playing) {{
            clearInterval(timer);
            playing = false;
            btn.innerHTML = '&#9654; Play';
        }} else {{
            playing = true;
            btn.innerHTML = '&#9724; Pause';
            timer = setInterval(function(){{
                var s = getActiveSlider();
                if (!s) return;
                var val = parseInt(s.value);
                var mx  = parseInt(s.max);
                if (val >= mx) {{
                    s.value = s.min;
                }} else {{
                    s.value = val + 1;
                }}
                s.dispatchEvent(new Event('input', {{bubbles: true}}));
            }}, 800);
        }}
    }});
}})();
</script>
"""
m3.get_root().html.add_child(folium.Element(play_js_dev))

# Save
out_path3 = os.path.join("..", "results", "figures", "gentrified_industries_local_median_mean_hotspot.html")
m3.save(out_path3)
print(f"Saved median/mean hotspot map to {out_path3}")
print(f"Clusters tracked: {len(styledict_median)}")
#m3

Thresholds: little ≤ 1, medium ≤ 3, high > 3
Saved median/mean hotspot map to ..\results\figures\gentrified_industries_local_median_mean_hotspot.html
Clusters tracked: 1186


## Map showing gentrified buinesses hotspot: Mean values global (Avg across all neighborhoods)

In [12]:
import folium
from folium.plugins import TimeSliderChoropleth
import datetime
import numpy as np

# --- Compute global mean count per year (average across ALL neighborhoods) ---
global_mean_per_year = full_df.groupby("year")["count"].mean()
print("Global mean count per year:")
print(global_mean_per_year.to_string())

# Compute deviation from the global mean for each cluster-year
full_df_global = full_df.copy()
full_df_global = full_df_global.merge(
    global_mean_per_year.rename("global_mean"), on="year"
)
full_df_global["dev_global"] = full_df_global["count"] - full_df_global["global_mean"]

# Build lookup: cluster_id → year → global deviation
dev_global_lookup = {}
for _, row in full_df_global.iterrows():
    dev_global_lookup.setdefault(row["cluster_id"], {})[row["year"]] = row["dev_global"]

# --- Colors & thresholds (same scheme as local map) ---
BLUE   = "#2166ac"
WHITE  = "#f7f7f7"
YELLOW = "#fee08b"
ORANGE = "#f46d43"
RED    = "#a50026"
GRAY   = "#d3d3d3"
t1 = 1
t2 = 3

def classify_color_g(dev, has_data):
    if not has_data:
        return GRAY
    if dev < 0:
        return BLUE
    if dev == 0:
        return WHITE
    if dev <= t1:
        return YELLOW
    if dev <= t2:
        return ORANGE
    return RED

def classify_label_g(dev):
    if dev < 0:
        return "Below"
    if dev == 0:
        return "At"
    if dev <= t1:
        return "Little above"
    if dev <= t2:
        return "Medium above"
    return "High above"

# --- Build styledict and popups ---
FRE_ID = "147_all"
styledict_global = {}
popup_dict_global = {}

for cid in clusters_4326["feature_id"]:
    styles = {}
    is_fre = (cid == FRE_ID)
    has_data = cid in dev_global_lookup
    popup_rows = []

    for year in years:
        epoch = str(int(datetime.datetime(year, 1, 1).timestamp()))
        if is_fre:
            styles[epoch] = {"color": GRAY, "opacity": 0.0}
            continue

        d = dev_global_lookup.get(cid, {}).get(year, 0.0)
        styles[epoch] = {"color": classify_color_g(d, has_data), "opacity": 0.8}

        cnt = full_df_global[(full_df_global["cluster_id"] == cid) & (full_df_global["year"] == year)]["count"].values[0]
        g_mean = full_df_global[(full_df_global["cluster_id"] == cid) & (full_df_global["year"] == year)]["global_mean"].values[0]
        cat = classify_label_g(d) if has_data else "N/A"
        popup_rows.append(
            f"<tr><td>{year}</td><td>{cnt}</td>"
            f"<td>{g_mean:.1f}</td><td>{d:.1f}</td><td>{cat}</td></tr>"
        )

    styledict_global[cid] = styles

    popup_html = (
        f"<b>Neighborhood: {cid}</b><br>"
        f"<table border='1' style='border-collapse:collapse;font-size:11px;'>"
        f"<tr><th>Year</th><th>Count</th>"
        f"<th>Global Mean</th><th>Deviation</th><th>Category</th></tr>"
        f"{''.join(popup_rows)}</table>"
    )
    popup_dict_global[cid] = popup_html

# --- Build GeoJSON ---
geo_data_g = clusters_4326[["feature_id", "geometry"]].copy().set_index("feature_id")
geo_json_g = geo_data_g.to_json()

# --- Create map ---
m4 = folium.Map(location=center, zoom_start=12, tiles="cartodbpositron")

tsc_global = TimeSliderChoropleth(
    data=geo_json_g, styledict=styledict_global,
    name="Global mean deviation",
)
tsc_global.add_to(m4)

# Popup layer
popup_gdf_g = clusters_4326[["feature_id", "geometry"]].copy()
popup_gdf_g["popup_html"] = popup_gdf_g["feature_id"].map(popup_dict_global)

folium.GeoJson(
    json.loads(popup_gdf_g.to_json()),
    style_function=lambda feat: {
        "fillOpacity": 0,
        "color": "black",
        "weight": 0.6,
    },
    popup=folium.GeoJsonPopup(
        fields=["popup_html"],
        aliases=[""],
        labels=False,
        parse_html=True,
        max_width=500,
    ),
    name="Cluster borders",
).add_to(m4)

# --- Legend ---
legend_html_g = f"""
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
    background:white; padding:10px 14px; border:2px solid grey; border-radius:5px;
    font-size:13px; line-height:1.6;">
<b>Deviation from global mean</b><br>
<i style="background:{BLUE};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> Below<br>
<i style="background:{WHITE};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> At<br>
<i style="background:{YELLOW};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> Little above (≤ 1)<br>
<i style="background:{ORANGE};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> Medium above (1–3)<br>
<i style="background:{RED};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> High above (> 3)<br>
<i style="background:{GRAY};width:14px;height:14px;display:inline-block;border:1px solid #999;"></i> No data
</div>
"""
m4.get_root().html.add_child(folium.Element(legend_html_g))

# --- Play / Pause button ---
play_js_g = """
<style>
#global-play-btn {
    position: fixed; bottom: 24px; left: 50%; transform: translateX(-50%);
    z-index: 10000; padding: 8px 22px; font-size: 15px; cursor: pointer;
    background: white; border: 2px solid #666; border-radius: 5px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.3);
}
</style>
<button id="global-play-btn">&#9654; Play</button>
<script>
(function(){
    var playing = false, timer = null;

    function getSlider() {
        var container = document.querySelector('[id^="slider_macro_element_"]');
        if (!container) return null;
        return container.querySelector('input[type="range"]');
    }

    document.getElementById('global-play-btn').addEventListener('click', function() {
        var btn = this;
        var slider = getSlider();
        if (!slider) {
            btn.textContent = 'Slider not found';
            return;
        }
        if (playing) {
            clearInterval(timer);
            playing = false;
            btn.innerHTML = '&#9654; Play';
        } else {
            playing = true;
            btn.innerHTML = '&#9724; Pause';
            timer = setInterval(function(){
                var val = parseInt(slider.value);
                var mx  = parseInt(slider.max);
                if (val >= mx) {
                    slider.value = slider.min;
                } else {
                    slider.value = val + 1;
                }
                slider.dispatchEvent(new Event('input', {bubbles: true}));
            }, 800);
        }
    });
})();
</script>
"""
m4.get_root().html.add_child(folium.Element(play_js_g))

# Save
out_path4 = os.path.join("..", "results", "figures", "gentrified_industries_global_mean_hotspot.html")
m4.save(out_path4)
print(f"Saved global mean hotspot map to {out_path4}")
print(f"Clusters tracked: {len(styledict_global)}")
#m4

Global mean count per year:
year
1987    0.260540
1990    0.332209
1993    0.454469
1996    0.584317
1999    0.803541
2002    1.049747
2005    1.301012
2008    1.744519
2011    2.241147
2014    2.703204
2017    3.780776
2020    4.531197
2023    4.231029
2026    3.847386
Saved global mean hotspot map to ..\results\figures\gentrified_industries_global_mean_hotspot.html
Clusters tracked: 1186
